
# Automated API Testing, Response Validation & Documentation Engine

## AI Agent with vLLM, PydanticAI

This notebook builds an AI-powered system to:
- Automatically test APIs
- Validate responses
- Generate cURL commands
- Create API documentation

---


## Step 1: Install Dependencies

In [ ]:
!pip install -q pydantic-ai-slim[openai] requests fastapi pandas

## Step 2: Set Environment Variables

In [ ]:
import os
BASE_URL = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'abc-123'

## Step 3: Initialize Model

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(base_url=BASE_URL, api_key='abc-123')
model = OpenAIChatModel('Qwen3-4B', provider=provider)

## Step 4: Create Agent

In [ ]:
from pydantic_ai import Agent
agent = Agent(model=model)

## Step 5: Helper Runner

In [ ]:
import asyncio
async def run_async(prompt):
    result = await agent.run(prompt)
    return result.output

## Step 6: API Execution Tool

In [ ]:
import requests
from pydantic_ai import Tool

@Tool
def execute_api(url: str, method: str, payload: dict):
    if method == 'GET':
        response = requests.get(url)
    else:
        response = requests.post(url, json=payload)
    return {'status': response.status_code, 'response': response.text}

## Step 7: Validation Tool

In [ ]:
@Tool
def validate_response(actual: dict, expected: dict):
    return {'result': 'PASS' if actual == expected else 'FAIL'}

## Step 8: CURL Generator

In [ ]:
@Tool
def generate_curl(url: str, method: str, payload: dict):
    return f'curl -X {method} {url} -d {payload}'

## Step 9: Documentation Tool

In [ ]:
@Tool
def generate_doc(url: str, payload: dict, response: dict):
    return {'endpoint': url, 'request': payload, 'response': response}

## Step 10: Enhance Agent

In [ ]:
agent = Agent(model=model, tools=[execute_api, validate_response, generate_curl, generate_doc])

## Step 11: Test Run

In [ ]:
result = await run_async('Test API https://jsonplaceholder.typicode.com/posts GET')
print(result)

## Step 12: Sample Report

In [ ]:
report = {'test': 'GET posts', 'status': 'PASS'}
print(report)